## Nitro NLP 2024
- the below version is:
    - tfidfvec + logistic regression model with xlmr finetune on the first task
    - same tfidf + LR combo

this implementations get 74/100

In [ ]:
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, FeatureUnion

In [ ]:
DS_DIR = "/kaggle/input/datasets/ababeiraul/msiddataset"

train_df = pd.read_csv(f"{DS_DIR}/train_data.csv")
test_df = pd.read_csv(f"{DS_DIR}/test_data.csv")


print(train_df.head())


In [ ]:
def clean_ro(text):
    return str(text).replace("ş", "ș").replace("ţ", "ț").replace("Ş", "Ș").replace("Ţ", "Ț")

train_df["sample"] = train_df["sample"].apply(clean_ro)
test_df["sample"] = test_df["sample"].apply(clean_ro)

In [ ]:
# refac etichete din 1 si 2 in 0 si 1 + le bag intr-un vector numpy (y)
y = (train_df["dialect"] == 2).astype(int).values

print(y)

In [ ]:
vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range = (1,1),
                            max_features = 30000,
                            sublinear_tf = True,
                            lowercase = False),
    ),
    ("char", TfidfVectorizer(analyzer = "char_wb",
                            ngram_range = (3, 5),
                            max_features = 50000,
                            sublinear_tf = True,
                            lowercase = False),
    )
])

In [ ]:

# we use the stratified version to keep the portions of the romanian/moldovian dialect the same in each group
strkf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
oof_probs_tfidf = np.zeros(len(train_df))
preds_probs_tfidf = np.zeros(len(test_df))


for fold, (tr_idx, val_idx) in enumerate(strkf.split(train_df, y)):
    X_tr = train_df["sample"].iloc[tr_idx]
    Y_tr = y[tr_idx];

    pipe = make_pipeline(
        vectorizer,
        LogisticRegression(C = 2.0, max_iter = 1000)
    )

    pipe.fit(X_tr, Y_tr)
    val_tr = train_df["sample"].iloc[val_idx]
    test_tr = test_df["sample"]
    
    oof_probs_tfidf[val_idx] = pipe.predict_proba(val_tr)[:, 1]
    preds_probs_tfidf += pipe.predict_proba(test_tr)[:, 1] / 5.0


In [ ]:
import torch
from transformers import  (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

In [ ]:
MODEL_NAME = "dumitrescustefan/bert-base-romanian-cased-v1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def tokenize_fn(examples):
    return tokenizer(examples["sample"], truncation = True, max_length = 192)

test_ds = Dataset.from_pandas(test_df[["sample"]]).map(
    tokenize_fn, 
    batched = True,
    remove_columns = ["sample"]
)

In [ ]:
oof_probs_xlmr = np.zeros(len(train_df))
preds_probs_xlmr = np.zeros(len(test_df))

def compute_metrics(eval_preds):
    logits, labels = eval_preds

    preds = np.argmax(logits, axis = 1)
    f1 = f1_score(labels, preds)

    return {
        "f1" : f1
    }


for fold, (tr_idx, val_idx) in enumerate(strkf.split(train_df, y)):
    
    tr_ds = Dataset.from_pandas(
        pd.DataFrame({"sample": train_df["sample"].iloc[tr_idx], "label": y[tr_idx]}).reset_index(drop = True)    
    ).map(
        tokenize_fn, 
        batched = True,
        remove_columns = ["sample"]
    )

    val_ds = Dataset.from_pandas(
        pd.DataFrame({"sample": train_df["sample"].iloc[val_idx], "label": y[val_idx]}).reset_index(drop = True)    
    ).map(
        tokenize_fn, 
        batched = True,
        remove_columns = ["sample"]
    )

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels = 2)
    args = TrainingArguments(
        output_dir="/kaggle/working/xlmr_checkpoint",
        num_train_epochs=4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=64,
        learning_rate=3e-5,
        weight_decay=0.01,           
        warmup_steps = 15,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model = "f1",
        greater_is_better=True,
        report_to="none"
    )

    trainer = Trainer(
        model,
        args = args,
        train_dataset = tr_ds,
        eval_dataset = val_ds,
        data_collator = DataCollatorWithPadding(tokenizer),
        compute_metrics = compute_metrics
    )

    trainer.train()

    val_logits = trainer.predict(val_ds).predictions
    oof_probs_xlmr[val_idx] = torch.softmax(torch.tensor(val_logits), dim = -1)[:, 1].numpy()

    test_logits = trainer.predict(test_ds).predictions
    preds_probs_xlmr = torch.softmax(torch.tensor(test_logits), dim = -1)[:, 1].numpy() / 5.0
    


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# evaluate on the 5 folds
print("5-Fold CV F1 doar TF-IDF :", round(f1_score(y, (oof_probs_tfidf >= 0.5).astype(int)), 4))
print("5-Fold CV F1 doar XLM-R  :", round(f1_score(y, (oof_probs_xlmr >= 0.5).astype(int)), 4))

best_w, best_f1 = 0.5, 0.0
for w in np.linspace(0, 1, 21):  # 0.00, 0.05, 0.10, ..., 1.00
    blend_oof = w * oof_probs_xlmr + (1 - w) * oof_probs_tfidf
    score = f1_score(y, (blend_oof >= 0.5).astype(int))
    if score > best_f1:
        best_f1, best_w = score, w

print(f"5-Fold CV F1 COMBINAT    : {best_f1:.4f} (greutate optima XLM-R = {best_w:.2f}, TF-IDF = {1 - best_w:.2f})")

In [ ]:
best_ts = 0.5

preds_blend = best_w * preds_probs_xlmr + (1 - best_w) * preds_probs_tfidf

task1_preds = np.where(preds_blend >= best_ts, 2, 1)



submission1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test_df["datapointID"],
    "answer": task1_preds
})


submission2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": test_df["datapointID"],
    "answer": 1
})


submission = pd.concat([submission1, submission2], ignore_index = True)
submission.to_csv("submission_task1_1.csv", index = False)

print(submission.head())
    

# from here its task 2 
- we try to give on the folds to 3 dif classifying models and then try to blend brute force
- 0.71 best f1 score blended => pretty bad 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score

In [ ]:
DS_DIR = "/kaggle/input/datasets/ababeiraul/msiddataset"

train_df = pd.read_csv(f"{DS_DIR}/train_data.csv")
test_df = pd.read_csv(f"{DS_DIR}/test_data.csv")


print(train_df.head())


In [ ]:
vec = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=60000, sublinear_tf=True, lowercase=False)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 6), max_features=80000, sublinear_tf=True, lowercase=False)),
])

In [ ]:
def clean_ro(s):
    return (
        s.fillna("").astype(str)
        .str.replace("ţ", "ț").str.replace("ş", "ș")
        .str.replace("Ţ", "Ț").str.replace("Ş", "Ș")
    )

X_train = clean_ro(train_df["sample"])
X_test = clean_ro(test_df["sample"])
y2 = (train_df["category"] - 1).values

In [ ]:
strkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_lr, test_lr = np.zeros((len(train_df), 6)), np.zeros((len(test_df), 6))
oof_svc, test_svc = np.zeros((len(train_df), 6)), np.zeros((len(test_df), 6))
oof_nb, test_nb = np.zeros((len(train_df), 6)), np.zeros((len(test_df), 6))

for tr_idx, val_idx in strkf.split(X_train, y2):
    X_tr = vec.fit_transform(X_train.iloc[tr_idx])
    X_val = vec.transform(X_train.iloc[val_idx])
    X_te = vec.transform(X_test)

    lr = LogisticRegression(C=1.5, max_iter=1000).fit(X_tr, y2[tr_idx])
    svc = CalibratedClassifierCV(LinearSVC(C=0.4), cv=3).fit(X_tr, y2[tr_idx])
    nb = ComplementNB(alpha=0.25).fit(X_tr, y2[tr_idx])

    oof_lr[val_idx] = lr.predict_proba(X_val)
    oof_svc[val_idx] = svc.predict_proba(X_val)
    oof_nb[val_idx] = nb.predict_proba(X_val)

    test_lr += lr.predict_proba(X_te) / 5.0
    test_svc += svc.predict_proba(X_te) / 5.0
    test_nb += nb.predict_proba(X_te) / 5.0

In [ ]:
oof_ens = 0.45 * oof_lr + 0.35 * oof_svc + 0.20 * oof_nb
test_ens = 0.45 * test_lr + 0.35 * test_svc + 0.20 * test_nb

print("CV Weighted F1 LR      :", round(f1_score(y2, np.argmax(oof_lr, axis=1), average="weighted"), 4))
print("CV Weighted F1 SVC     :", round(f1_score(y2, np.argmax(oof_svc, axis=1), average="weighted"), 4))
print("CV Weighted F1 NB      :", round(f1_score(y2, np.argmax(oof_nb, axis=1), average="weighted"), 4))
print("CV Weighted F1 ENSEMBLE:", round(f1_score(y2, np.argmax(oof_ens, axis=1), average="weighted"), 4))

task2_preds = np.argmax(test_ens, axis=1) + 1

In [ ]:
import pandas as pd


submission2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": test_df["datapointID"],
    "answer": task2_preds
})


submission2.to_csv("submission_task1_2.csv", index=False)

print(submission2.head())
